In [ ]:
import tonic
import tonic.transforms as T
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader


import snntorch as snn
from snntorch import utils
from snntorch import surrogate

transform = T.NumpyAsType(int)
dataset = tonic.datasets.NMNIST(save_to="./data", train=False, transform=transform) # time, number of polarities, height, width

sensor_size = dataset.sensor_size
print("Sensor size:", sensor_size) #x, y, number of polarities


In [ ]:
# super basic spiking neural network
class BasicSNN(nn.Module):
    def __init__(self):
        super(BasicSNN, self).__init__()
        spike_grad = surrogate.atan()
        self.fc1 = nn.Linear(sensor_size[0] * sensor_size[1], 256)
        self.lif1 = snn.Leaky(beta=0.9, spike_grad=spike_grad)
        self.fc2 = nn.Linear(256, 10)
        self.lif2 = snn.Leaky(beta=0.9, spike_grad=spike_grad)


    def forward(self, data):
        # x will be a tensor, carrying (x, y, time, polarity)
        mem1 = self.lif1.reset_mem()
        mem2 = self.lif2.reset_mem()
        outputs = []

        for event in data:
            x, y, t, p = event
            # negative polarity represented with -1 instead of 0, as we're using a sparse representation
            p = -1 if p == 0 else p 
            # create sparse representation
            idx = y * sensor_size[1] + x
            indices = torch.tensor([[idx]], device=data.device)
            values = torch.tensor([p], device=data.device, dtype=torch.float32)
            input_tensor = torch.sparse_coo_tensor(indices=indices, values=values, size=(sensor_size[0] * sensor_size[1],), device=data.device)

            cur1 = self.fc1(input_tensor)
            spk1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            outputs.append(spk2)

        return torch.stack(outputs, dim=0)
            

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# enforced to be 1, as samples have varying lengths
# Can probably do some padding and create a workaround, nothing at this stage
batch_size = 1
num_epochs = 10

model = BasicSNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for i, (events, label) in enumerate(train_loader):
        events = events.squeeze(0)
        label_idx = label.item()
        label = F.one_hot(label, num_classes=10).float()
        events = events.to(device)
        label = label.to(device)
        optimizer.zero_grad()
        # print(f"events shape: {events.shape}, label shape: {label.shape}")

        spk_rec = model(events)
        # Sum outputs across time (rate coding)
        final_output = spk_rec.mean(dim=0)
        loss = criterion(final_output.unsqueeze(0), label)
        # print(f"final_output: {final_output}, label: {label}")
        loss.backward()
        optimizer.step()
        # Update statistics
        train_loss += loss.item()
        predicted_idx = final_output.argmax().item()
        total += batch_size
        correct += (predicted_idx == label_idx)
        
        utils.reset(model)
    
    # Print statistics once per epoch
    print(f'Epoch: {epoch+1}, Loss: {train_loss/len(train_loader):.4f}, '
          f'Accuracy: {100.*correct/total:.2f}%')
    